# Data Preprocessing Agent
**Section I — Data Understanding + Hypothesis Setup (Joey)**

**Builds the first usable modeling table from raw X5 data.**

Steps:
1. Load ingestion artifacts (`clients`, `purchases`, labeled splits)
2. Filter purchases to **pre-treatment window only** (core anti-leakage step)
3. Compute RFM features (Recency, Frequency, Monetary + extensions)
4. Process demographic features (age, gender)
5. Join all features to one row per `client_id`
6. Validate: no duplicates, binary labels, no leakage columns
7. Save `modeling_table_{split}.parquet` for the experiment pipeline

---
**Leakage rule:** Features may only use purchases dated **strictly before** each
customer's `first_issue_date`. `first_redeem_date` is post-treatment and is **never** used.

In [ ]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings('ignore')

ARTIFACTS_DIR = 'artifacts'
OUTPUT_DIR    = 'artifacts'

In [ ]:
train_df  = pd.read_parquet(f'{ARTIFACTS_DIR}/train_split.parquet')
val_df    = pd.read_parquet(f'{ARTIFACTS_DIR}/val_split.parquet')
test_df   = pd.read_parquet(f'{ARTIFACTS_DIR}/test_split.parquet')
clients   = pd.read_parquet(f'{ARTIFACTS_DIR}/clients.parquet')
purchases = pd.read_parquet(f'{ARTIFACTS_DIR}/purchases.parquet')

purchases['transaction_datetime'] = pd.to_datetime(purchases['transaction_datetime'])
for df in [train_df, val_df, test_df]:
    if 'first_issue_date' in df.columns:
        df['first_issue_date'] = pd.to_datetime(df['first_issue_date'])

print("Loaded artifacts:")
print(f"  train_split : {len(train_df):>8,} rows")
print(f"  val_split   : {len(val_df):>8,} rows")
print(f"  test_split  : {len(test_df):>8,} rows")
print(f"  clients     : {len(clients):>8,} rows")
print(f"  purchases   : {len(purchases):>8,} rows")
print(f"\nTrain cols: {list(train_df.columns)}")

## 1. Pre-Treatment Purchase Filtering

Each customer has a `first_issue_date` — the date their coupon was issued.

We keep only purchases **strictly before** that date per customer. This is the
primary anti-leakage control: the model sees only what was knowable before contact.

Customers with **no** pre-treatment purchase history are kept with zero-filled
RFM features and `has_purchase_history = 0`.

In [ ]:
all_labeled = pd.concat([
    train_df.assign(split='train'),
    val_df.assign(split='val'),
    test_df.assign(split='test'),
], ignore_index=True)

# Attach each labeled client's treatment date to their purchases (inner join = labeled only)
purchases_labeled = purchases.merge(
    all_labeled[['client_id', 'first_issue_date']],
    on='client_id',
    how='inner',
)

# Keep only pre-treatment purchases
pre_tx = purchases_labeled[
    purchases_labeled['transaction_datetime'] < purchases_labeled['first_issue_date']
].copy()

n_total = len(purchases_labeled)
n_kept  = len(pre_tx)
n_drop  = n_total - n_kept

print(f"Purchases for labeled clients      : {n_total:>10,}")
print(f"Pre-treatment purchases (kept)     : {n_kept:>10,}  ({n_kept/n_total:.1%})")
print(f"Post/on-treatment (filtered out)   : {n_drop:>10,}  ({n_drop/n_total:.1%})")
print(f"\nClients with purchase history      : {pre_tx['client_id'].nunique():,} / {len(all_labeled):,} ({pre_tx['client_id'].nunique()/len(all_labeled):.1%})")

## 2. RFM Feature Engineering

| Feature | Description |
|---|---|
| `recency_days` | Days since last pre-treatment purchase relative to treatment date. `-1` if no history. |
| `frequency` | Number of unique transactions |
| `monetary_total` | Total pre-treatment spend |
| `monetary_avg` | Average spend per transaction |
| `n_stores_visited` | Distinct stores visited (shopping breadth) |
| `n_unique_products` | Distinct products purchased |
| `avg_qty_per_tx` | Average quantity per transaction (basket size proxy) |
| `purchase_span_days` | Days between first and last pre-treatment purchase |
| `has_purchase_history` | 1 if any pre-treatment purchases exist, else 0 |

**Hypothesis link:** Persuadable customers are expected to have moderate-to-high frequency
and monetary values, moderate recency, and respond positively to a coupon nudge.
Lost causes and sure things should be separable via these features + the uplift model.

In [ ]:
rfm_agg = (
    pre_tx
    .groupby('client_id')
    .agg(
        last_purchase_date  = ('transaction_datetime', 'max'),
        first_purchase_date = ('transaction_datetime', 'min'),
        frequency           = ('transaction_id',       'nunique'),
        monetary_total      = ('purchase_sum',         'sum'),
        monetary_avg        = ('purchase_sum',         'mean'),
        n_stores_visited    = ('store_id',             'nunique'),
        n_unique_products   = ('product_id',           'nunique'),
        avg_qty_per_tx      = ('product_quantity',     'mean'),
    )
    .reset_index()
)

# Attach treatment date for recency computation (left join to keep all labeled clients)
rfm = all_labeled[['client_id', 'first_issue_date']].merge(rfm_agg, on='client_id', how='left')

rfm['recency_days']         = (rfm['first_issue_date'] - rfm['last_purchase_date']).dt.days
rfm['purchase_span_days']   = (rfm['last_purchase_date'] - rfm['first_purchase_date']).dt.days
rfm['has_purchase_history'] = rfm['last_purchase_date'].notna().astype(int)

# Zero-fill for clients with no pre-treatment history
fill_cols = ['frequency', 'monetary_total', 'monetary_avg',
             'n_stores_visited', 'n_unique_products', 'avg_qty_per_tx', 'purchase_span_days']
rfm[fill_cols] = rfm[fill_cols].fillna(0)
rfm['recency_days'] = rfm['recency_days'].fillna(-1)

rfm = rfm.drop(columns=['first_issue_date', 'last_purchase_date', 'first_purchase_date'])

print(f"RFM features shape: {rfm.shape}")
print(f"Clients with no pre-treatment history: {(rfm['frequency'] == 0).sum():,} ({(rfm['frequency'] == 0).mean():.1%})")
print("\nRFM summary (active clients only):")
rfm[rfm['has_purchase_history'] == 1].describe().round(2)

## 3. Demographic Feature Processing

In [ ]:
client_feats = clients[['client_id']].copy()

# Age: coerce to numeric (may be stored as category string in some versions)
if 'age' in clients.columns:
    client_feats['age'] = pd.to_numeric(clients['age'], errors='coerce')

# Gender: M=1, F=0, unknown=-1
if 'gender' in clients.columns:
    gender_map = {'M': 1, 'F': 0, 'U': -1, 'u': -1}
    client_feats['gender'] = clients['gender'].map(gender_map).fillna(-1).astype(int)

print(f"Client feature table: {client_feats.shape}")
if 'gender' in client_feats.columns:
    print(f"\nGender distribution (1=M, 0=F, -1=unknown):\n{client_feats['gender'].value_counts().to_string()}")
if 'age' in client_feats.columns:
    age_null_pct = client_feats['age'].isna().mean()
    print(f"\nAge null rate: {age_null_pct:.2%}")
    print(client_feats['age'].describe().round(1).to_string())

## 4. Build Final Modeling Table

In [ ]:
modeling_table = (
    all_labeled[['client_id', 'split', 'target', 'treatment']]
    .merge(client_feats, on='client_id', how='left')
    .merge(rfm,          on='client_id', how='left')
)

print(f"Modeling table: {modeling_table.shape[0]:,} rows x {modeling_table.shape[1]} columns")
print(f"Columns: {list(modeling_table.columns)}")

## 5. Leakage & Quality Validation

In [ ]:
print("Running validation checks...")

# One row per client
n_dup = modeling_table['client_id'].duplicated().sum()
assert n_dup == 0, f"Duplicate client_ids: {n_dup}"
print("\u2713 No duplicate client_ids")

# Binary labels
assert modeling_table['target'].isin([0, 1]).all(),    "Non-binary target"
assert modeling_table['treatment'].isin([0, 1]).all(), "Non-binary treatment"
print("\u2713 target and treatment are binary")

# No nulls in labels
assert modeling_table['target'].notna().all(),    "Nulls in target"
assert modeling_table['treatment'].notna().all(), "Nulls in treatment"
print("\u2713 No nulls in target/treatment")

# No post-treatment leakage columns (redeem date)
leakage_cols = [c for c in modeling_table.columns if 'redeem' in c.lower()]
assert not leakage_cols, f"Post-treatment leakage columns found: {leakage_cols}"
print("\u2713 No post-treatment leakage columns detected")

# Distribution check across splits
print("\nDistribution by split:")
print(f"  {'split':<8} {'n':>8} {'treatment':>12} {'target':>9}")
for split in ['train', 'val', 'test']:
    sub = modeling_table[modeling_table['split'] == split]
    print(f"  {split:<8} {len(sub):>8,} {sub['treatment'].mean():>12.4f} {sub['target'].mean():>9.4f}")

# Feature null summary
feat_cols = [c for c in modeling_table.columns if c not in ['client_id', 'split', 'target', 'treatment']]
high_null = modeling_table[feat_cols].isnull().mean()
high_null = high_null[high_null > 0.05]
if high_null.empty:
    print("\n\u2713 All features have < 5% nulls")
else:
    print("\n\u2139 Features with >5% nulls (may need imputation in training):")
    for col, pct in high_null.items():
        print(f"  {col:<35} {pct:.2%}")

print("\n\u2713 All validation checks passed")

## 6. Save Modeling Tables

In [ ]:
for split in ['train', 'val', 'test']:
    subset = modeling_table[modeling_table['split'] == split].drop(columns=['split'])
    out_path = f"{OUTPUT_DIR}/modeling_table_{split}.parquet"
    subset.to_parquet(out_path, index=False)
    print(f"\u2713 Saved {out_path} -- {len(subset):,} rows x {subset.shape[1]} feature cols")

In [ ]:
feat_cols = [c for c in modeling_table.columns if c not in ['client_id', 'split', 'target', 'treatment']]

print("=" * 55)
print("DATA PREPROCESSING AGENT - SUMMARY REPORT")
print("=" * 55)
print(f"Total labeled clients  : {len(modeling_table):,}")
print(f"Feature columns        : {len(feat_cols)}")
demo_cols = [c for c in feat_cols if c in ['age', 'gender']]
rfm_cols  = [c for c in feat_cols if c not in ['age', 'gender']]
print(f"  Demographics         : {demo_cols}")
print(f"  RFM + extensions     : {rfm_cols}")
print(f"\nLeakage prevention: pre-treatment filter applied \u2713")
print(f"One row per client_id  : \u2713")
print(f"\nSaved modeling tables:")
for split in ['train', 'val', 'test']:
    n = (modeling_table['split'] == split).sum()
    print(f"  modeling_table_{split}.parquet  {n:>8,} rows")
print("=" * 55)